In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Configuração da SparkSession (com suporte Hive)
warehouse_location = 'hdfs://hdfs-nn:9000/demo/Delta'
table_name = "titles_text_table_fixed"
data_location = f"{warehouse_location}/titles_text_data/" # Novo local para os dados

spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Titles CSV to Hive Text Table") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

print("SparkSession com suporte Hive iniciada.")

# 2. Ler o Ficheiro CSV do HDFS (da sua zona Bronze)
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/titles.csv"

movies_df = spark.read.csv(
    hdfs_path,
    header=True,        
    inferSchema=True,   
    multiLine=True,     
    escape='"'          
)

print(f"Schema do DataFrame '{table_name}' (inferido):")
movies_df.printSchema()

# 3. Guardar os dados CSV (texto) no HDFS com o separador TAB
print(f"\n3. A gravar dados CSV (TAB separado) em: {data_location}")

# Guardamos os dados no HDFS (não registamos a tabela aqui, apenas os dados)
movies_df.write \
    .format("csv") \
    .mode("overwrite") \
    .option("sep", "\t") \
    .save(data_location)

print("Dados CSV gravados com sucesso.")

# 3.5. REGISTAR A TABELA NO HIVE (CORREÇÃO CRUCIAL)
# Usamos o comando SQL para criar a tabela externa e DEFINIR o separador TAB (\t)
# O Hive usará agora o separador correto para ler os dados.
spark.sql("CREATE DATABASE IF NOT EXISTS demo")
spark.sql(f"DROP TABLE IF EXISTS demo.{table_name}")

print(f"\nA criar a definição da tabela '{table_name}' no Hive...")

spark.sql(f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS demo.{table_name} (
        id STRING,
        title STRING,
        type STRING,
        description STRING,
        release_year INT,
        age_certification STRING,
        runtime INT,
        genres STRING,
        production_countries STRING,
        seasons DOUBLE,
        imdb_id STRING,
        imdb_score DOUBLE,
        imdb_votes DOUBLE,
        tmdb_popularity DOUBLE,
        tmdb_score DOUBLE
    )
    ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe'
    WITH SERDEPROPERTIES ('field.delim' = '\\t')
    STORED AS TEXTFILE
    LOCATION '{data_location}'
""")


SparkSession com suporte Hive iniciada.
Schema do DataFrame 'titles_text_table_fixed' (inferido):
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)


3. A gravar dados CSV (TAB separado) em: hdfs://hdfs-nn:9000/demo/Delta/titles_text_data/
Dados CSV gravados com sucesso.

A criar a definição da tabela 'titles_text_table_fixed' no Hive...
Tabela 'titles_text_table_fixed' criada com sucesso com o separador correto (TAB

In [2]:
spark.sql(
    """
    SELECT * FROM demo.titles_text_table
    """
).show()

+--------+--------------------+-----+--------------------+------------+-----------------+-------+--------------------+--------------------+-------+---------+----------+----------+---------------+----------+
|      id|               title| type|         description|release_year|age_certification|runtime|              genres|production_countries|seasons|  imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|
+--------+--------------------+-----+--------------------+------------+-----------------+-------+--------------------+--------------------+-------+---------+----------+----------+---------------+----------+
|ts300399|Five Came Back: T...| SHOW|This collection i...|        1945|            TV-MA|     51|   ['documentation']|              ['US']|    1.0|     null|      null|      null|            0.6|      null|
| tm84618|         Taxi Driver|MOVIE|A mentally unstab...|        1976|                R|    114|  ['drama', 'crime']|              ['US']|   null|tt0075314|       8.2|  80

In [3]:
from pyspark.sql import SparkSession
# (Assumindo que o 'spark' já foi iniciado)

# O caminho HDFS para o seu ficheiro
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/titles.csv"

# A forma CORRETA de ler este CSV complexo:
movies_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(hdfs_path)

print("Schema Correto (Inferido pelo Spark):")
movies_df.printSchema()

print("Dados do Filme:")
movies_df.show(5)

Schema Correto (Inferido pelo Spark):
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

Dados do Filme:
+--------+--------------------+-----+--------------------+------------+-----------------+-------+--------------------+--------------------+-------+---------+----------+----------+---------------+----------+
|      id|               title| type|         description|release_year|age_certification|runtime|           

In [4]:
movies_df \
    .write \
    .mode("overwrite") \
    .format("csv") \
    .option("delimiter",";")\
    .save("hdfs://hdfs-nn:9000/demo/Delta/titles_silver/")

In [5]:
spark.stop()